<a href="https://colab.research.google.com/github/Tauhid-Topu-007/Thesis-4-1/blob/main/RCNN_Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchinfo

In [2]:
!mkdir ~/.kaggle

In [3]:
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [4]:
!kaggle datasets download sharansmenon/aquarium-dataset

Dataset URL: https://www.kaggle.com/datasets/sharansmenon/aquarium-dataset
License(s): copyright-authors
100% 68.0M/68.0M [00:04<00:00, 16.5MB/s]



In [5]:
!unzip /content/aquarium-dataset.zip -d /content/

Archive:  /content/aquarium-dataset.zip
  inflating: /content/Aquarium Combined/README.dataset.txt  
  inflating: /content/Aquarium Combined/README.roboflow.txt  
  inflating: /content/Aquarium Combined/test/IMG_2289_jpeg_jpg.rf.fe2a7a149e7b11f2313f5a7b30386e85.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2301_jpeg_jpg.rf.2c19ae5efbd1f8611b5578125f001695.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2319_jpeg_jpg.rf.6e20bf97d17b74a8948aa48776c40454.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2347_jpeg_jpg.rf.7c71ac4b9301eb358cd4a832844dedcb.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2354_jpeg_jpg.rf.396e872c7fb0a95e911806986995ee7a.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2371_jpeg_jpg.rf.54505f60b6706da151c164188c305849.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2379_jpeg_jpg.rf.7dc3160c937072d26d4624c6c48e904d.jpg  
  inflating: /content/Aquarium Combined/test/IMG_2380_jpeg_jpg.rf.a23809682eb1466c1136ca0f55de8fb5.jpg

In [6]:
# ============================================
# BLOCK 1: Required Libraries & Configuration
# ============================================

import os
import warnings
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from typing import List, Tuple, Dict
from collections import Counter

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter

# Torchvision
import torchvision
from torchvision import models
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import RPNHead
from torchvision.ops import nms

# Albumentations (Data Augmentation)
import albumentations as A
from albumentations.pytorch import ToTensorV2

# PyCOCOtools
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Utilities
from tqdm import tqdm
import torchinfo
from torchinfo import summary

# Suppress warnings
warnings.filterwarnings("ignore")
%matplotlib inline

# ============================================
# CONFIGURATION CLASS (All hyperparameters in one place)
# ============================================

class Config:
    # Data
    DATASET_PATH = "/content/Aquarium Combined"
    IMG_SIZE = 640  # 640x640 resolution (better than baseline)

    # Model
    BACKBONE = 'resnet50'  # 'mobilenet_v3', 'resnet50', 'resnet101'
    PRETRAINED = True

    # Anchor Generation (Custom for Aquarium dataset)
    ANCHOR_SIZES = ((16, 32, 64, 128, 256),)  # Small to large objects
    ASPECT_RATIOS = ((0.5, 1.0, 2.0),)

    # RPN Parameters
    RPN_NMS_THRESH = 0.7
    RPN_PRE_NMS_TOP_N_TRAIN = 2000
    RPN_PRE_NMS_TOP_N_TEST = 1000
    RPN_POST_NMS_TOP_N_TRAIN = 1000
    RPN_POST_NMS_TOP_N_TEST = 300

    # ROI Head Parameters
    ROI_OUTPUT_SIZE = 14  # Default 7, larger = better for small objects
    ROI_NMS_THRESH = 0.5
    ROI_SCORE_THRESH = 0.05
    ROI_BATCH_SIZE_PER_IMAGE = 512
    ROI_POSITIVE_FRACTION = 0.25

    # Training
    BATCH_SIZE = 4
    EPOCHS = 15
    LEARNING_RATE = 0.005
    MOMENTUM = 0.9
    WEIGHT_DECAY = 0.0005

    # Learning Rate Scheduler
    LR_SCHEDULER = 'cosine'  # 'step', 'cosine', 'reduce_on_plateau'
    STEP_SIZE = 5
    GAMMA = 0.1

    # Gradient Clipping
    GRAD_CLIP_NORM = 1.0

    # AMP (Mixed Precision)
    USE_AMP = True

    # DataLoader
    NUM_WORKERS = 2
    PIN_MEMORY = True

    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Seed for reproducibility
    SEED = 42

    @staticmethod
    def set_seed():
        """Set all seeds for reproducibility"""
        random.seed(Config.SEED)
        np.random.seed(Config.SEED)
        torch.manual_seed(Config.SEED)
        torch.cuda.manual_seed_all(Config.SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Set seed for reproducibility
Config.set_seed()
print("Configuration loaded successfully!")
print(f"Device: {Config.DEVICE}")
print(f"Dataset path: {Config.DATASET_PATH}")
print(f"Image size: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"Backbone: {Config.BACKBONE}")
print(f"Epochs: {Config.EPOCHS}, Batch Size: {Config.BATCH_SIZE}")

Configuration loaded successfully!
Device: cuda
Dataset path: /content/Aquarium Combined
Image size: 640x640
Backbone: resnet50
Epochs: 15, Batch Size: 4


In [7]:
# ============================================
# BLOCK 2 FIXED: Data Transformations and Augmentation
# ============================================

def get_train_transforms():
    """
    Returns transformation pipeline for training images with advanced augmentations
    including resizing, flipping, rotation, brightness/contrast adjustments,
    color jitter, noise, and shadow effects.
    """
    return A.Compose([
        # Resize images to configured size
        A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),

        # Geometric augmentations
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.1,
            scale_limit=0.15,
            rotate_limit=15,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            p=0.5
        ),

        # Color augmentations
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.3
        ),
        A.HueSaturationValue(
            hue_shift_limit=20,
            sat_shift_limit=30,
            val_shift_limit=20,
            p=0.3
        ),
        A.ColorJitter(
            brightness=0.1,
            contrast=0.1,
            saturation=0.1,
            hue=0.05,
            p=0.2
        ),

        # Noise and blur (for robustness)
        A.GaussNoise(
            var_limit=(10.0, 50.0),
            p=0.2
        ),
        A.Blur(
            blur_limit=3,
            p=0.1
        ),

        # Shadow augmentation
        A.RandomShadow(
            shadow_roi=(0, 0.5, 1, 1),
            num_shadows_lower=1,
            num_shadows_upper=2,
            p=0.2
        ),

        # Convert to tensor (no normalization here, we'll do it in dataset)
        ToTensorV2()
    ],
    bbox_params=A.BboxParams(
        format='coco',
        min_visibility=0.3  # Ignore boxes with <30% visibility after augmentation
    ))


def get_test_transforms():
    """
    Returns transformation pipeline for test/validation images.
    Only resizing and tensor conversion (no augmentations).
    """
    return A.Compose([
        A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
        ToTensorV2()
    ],
    bbox_params=A.BboxParams(
        format='coco'
    ))


def get_tta_transforms():
    """
    Returns Test Time Augmentation (TTA) transforms for better inference.
    Multiple augmentations to get robust predictions.
    """
    return [
        A.Compose([A.Resize(Config.IMG_SIZE, Config.IMG_SIZE), ToTensorV2()]),
        A.Compose([A.Resize(Config.IMG_SIZE, Config.IMG_SIZE), A.HorizontalFlip(p=1.0), ToTensorV2()]),
        A.Compose([A.Resize(Config.IMG_SIZE, Config.IMG_SIZE), A.VerticalFlip(p=1.0), ToTensorV2()]),
        A.Compose([A.Resize(Config.IMG_SIZE, Config.IMG_SIZE), A.RandomRotate90(p=1.0), ToTensorV2()]),
    ]


# Test the transforms
print("Testing transforms...")
train_transform = get_train_transforms()
test_transform = get_test_transforms()

# Create a dummy image and boxes
dummy_image = np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8)
dummy_boxes = [[100, 100, 50, 50]]  # x, y, w, h (no label needed for bbox_params)

# Test train transform
try:
    transformed = train_transform(image=dummy_image, bboxes=dummy_boxes)
    print("Train transform successful!")
    print(f"  Image shape: {transformed['image'].shape}")
    print(f"  Boxes: {transformed['bboxes']}")
except Exception as e:
    print(f"Error in train transform: {e}")

# Test test transform
try:
    transformed = test_transform(image=dummy_image, bboxes=dummy_boxes)
    print("Test transform successful!")
    print(f"  Image shape: {transformed['image'].shape}")
    print(f"  Boxes: {transformed['bboxes']}")
except Exception as e:
    print(f"Error in test transform: {e}")

print("\nData transformations ready!")

Testing transforms...
Train transform successful!
  Image shape: torch.Size([3, 640, 640])
  Boxes: [[213.3333396911621, 400.0, 106.66666030883789, 80.0]]
Test transform successful!
  Image shape: torch.Size([3, 640, 640])
  Boxes: [[160.0, 213.3333396911621, 80.0, 106.66666030883789]]

Data transformations ready!


In [8]:
# ============================================
# BLOCK 3: Custom Dataset Class for Aquarium Detection
# ============================================

class AquariumDetection(Dataset):
    """
    Custom Dataset class for Aquarium Object Detection.
    Handles COCO format annotations with proper transformations.
    """

    def __init__(self, root, split='train', transforms=None):
        """
        Initializes dataset with root path, split (train/test/valid), and transformations.
        Loads COCO annotations and filters images with annotations.

        Parameters:
        - root: path to the dataset
        - split: 'train', 'test', or 'valid' to select dataset split
        - transforms: data augmentation transformations
        """
        self.root = root
        self.split = split
        self.transforms = transforms

        # Load COCO format annotations
        annotation_path = os.path.join(root, split, "_annotations.coco.json")
        self.coco = COCO(annotation_path)

        # Filter image IDs that have at least one annotation
        self.ids = [img_id for img_id in sorted(self.coco.imgs.keys())
                    if len(self.coco.getAnnIds(img_id)) > 0]

        # Create class mapping
        self.classes = {cat['id']: cat['name'] for cat in self.coco.cats.values()}

        # Create reverse mapping for inference
        self.classes_reverse = {v: k for k, v in self.classes.items()}

        print(f"Dataset initialized: {split} split")
        print(f"  Total images with annotations: {len(self.ids)}")
        print(f"  Total classes: {len(self.classes)}")
        print(f"  Classes: {list(self.classes.values())}")

    def __getitem__(self, index):
        """
        Retrieves image and target (bounding boxes and labels) by index.

        Parameters:
        - index: index to select image and target

        Returns:
        - Transformed image tensor and target dictionary
        """
        img_id = self.ids[index]
        image = self.load_image(img_id)
        annotations = self.coco.loadAnns(self.coco.getAnnIds(img_id))

        # Extract bounding boxes and labels
        boxes = []
        labels = []
        for ann in annotations:
            x, y, w, h = ann['bbox']
            # Convert from COCO format (x, y, w, h) to (x_min, y_min, x_max, y_max)
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

        # Convert to numpy arrays for albumentations
        boxes = np.array(boxes, dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)

        # Apply transformations if specified
        if self.transforms:
            transformed = self.transforms(image=image, bboxes=boxes)
            image = transformed['image']
            boxes = np.array(transformed['bboxes'], dtype=np.float32)

        # Convert to tensors
        boxes = torch.from_numpy(boxes).float()
        labels = torch.from_numpy(labels).long()

        # Get area of boxes
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        # Get iscrowd (if available, default to 0)
        iscrowd = torch.zeros((len(annotations),), dtype=torch.int64)

        # Prepare target dictionary
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([img_id], dtype=torch.int64),
            'area': area,
            'iscrowd': iscrowd
        }

        # Normalize image to [0, 1] range
        image = image.float() / 255.0

        return image, target

    def load_image(self, img_id):
        """
        Loads image from file path using OpenCV and converts to RGB.

        Parameters:
        - img_id: image ID to load

        Returns:
        - Loaded RGB image as numpy array
        """
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.root, self.split, img_info['file_name'])
        image = cv2.imread(img_path)

        if image is None:
            raise ValueError(f"Could not load image: {img_path}")

        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

    def __len__(self):
        """Returns the total number of images with annotations in the dataset."""
        return len(self.ids)


def create_dataloaders():
    """
    Creates train, test, and validation dataloaders.

    Returns:
    - train_loader: DataLoader for training
    - test_loader: DataLoader for testing
    - valid_loader: DataLoader for validation
    - dataset: Training dataset (for accessing class information)
    """
    # Create datasets
    print("Creating datasets...")

    train_dataset = AquariumDetection(
        root=Config.DATASET_PATH,
        split='train',
        transforms=get_train_transforms()
    )

    test_dataset = AquariumDetection(
        root=Config.DATASET_PATH,
        split='test',
        transforms=get_test_transforms()
    )

    valid_dataset = AquariumDetection(
        root=Config.DATASET_PATH,
        split='valid',
        transforms=get_test_transforms()
    )

    # Create dataloaders
    print("\nCreating dataloaders...")

    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.PIN_MEMORY,
        drop_last=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.PIN_MEMORY
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=Config.NUM_WORKERS,
        pin_memory=Config.PIN_MEMORY
    )

    print(f"\nDataloaders created successfully!")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Test batches: {len(test_loader)}")
    print(f"  Valid batches: {len(valid_loader)}")

    return train_loader, test_loader, valid_loader, train_dataset


# Create dataloaders
train_loader, test_loader, valid_loader, train_dataset = create_dataloaders()

# Print class mapping
print("\nClass mapping:")
for class_id, class_name in train_dataset.classes.items():
    print(f"  {class_id}: {class_name}")

Creating datasets...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Dataset initialized: train split
  Total images with annotations: 447
  Total classes: 8
  Classes: ['creatures', 'fish', 'jellyfish', 'penguin', 'puffin', 'shark', 'starfish', 'stingray']
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Dataset initialized: test split
  Total images with annotations: 63
  Total classes: 8
  Classes: ['creatures', 'fish', 'jellyfish', 'penguin', 'puffin', 'shark', 'starfish', 'stingray']
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Dataset initialized: valid split
  Total images with annotations: 127
  Total classes: 8
  Classes: ['creatures', 'fish', 'jellyfish', 'penguin', 'puffin', 'shark', 'starfish', 'stingray']

Creating dataloaders...

Dataloaders created successfully!
  Train batches: 112
  Test batches: 16
  Valid batches: 32

Class mapping:
  0: creatures
  1: fish
  2:

In [9]:
# ============================================
# BLOCK 4: Model Creation and Configuration
# ============================================

def create_model(num_classes):
    """
    Creates and configures Faster R-CNN model with custom settings.

    Parameters:
    - num_classes: Number of classes (including background)

    Returns:
    - Configured Faster R-CNN model
    """
    print("Creating Faster R-CNN model...")
    print(f"  Number of classes: {num_classes}")

    # Select backbone based on configuration
    if Config.BACKBONE == 'mobilenet_v3':
        print("  Using MobileNetV3 backbone (lightweight)")
        model = models.detection.fasterrcnn_mobilenet_v3_large_fpn(
            pretrained=Config.PRETRAINED
        )
    elif Config.BACKBONE == 'resnet50':
        print("  Using ResNet50 backbone (balanced)")
        model = models.detection.fasterrcnn_resnet50_fpn(
            pretrained=Config.PRETRAINED,
            pretrained_backbone=True
        )
    elif Config.BACKBONE == 'resnet101':
        print("  Using ResNet101 backbone (heavy - more accurate)")
        model = models.detection.fasterrcnn_resnet101_fpn(
            pretrained=Config.PRETRAINED,
            pretrained_backbone=True
        )
    else:
        raise ValueError(f"Unknown backbone: {Config.BACKBONE}")

    # Custom anchor generator for Aquarium dataset
    anchor_generator = AnchorGenerator(
        sizes=Config.ANCHOR_SIZES,
        aspect_ratios=Config.ASPECT_RATIOS
    )
    model.rpn.anchor_generator = anchor_generator
    print(f"  Custom anchors: {Config.ANCHOR_SIZES}")

    # Configure RPN parameters
    model.rpn.nms_thresh = Config.RPN_NMS_THRESH
    model.rpn.pre_nms_top_n_train = Config.RPN_PRE_NMS_TOP_N_TRAIN
    model.rpn.pre_nms_top_n_test = Config.RPN_PRE_NMS_TOP_N_TEST
    model.rpn.post_nms_top_n_train = Config.RPN_POST_NMS_TOP_N_TRAIN
    model.rpn.post_nms_top_n_test = Config.RPN_POST_NMS_TOP_N_TEST
    print(f"  RPN NMS threshold: {Config.RPN_NMS_THRESH}")
    print(f"  RPN pre-NMS top N (train): {Config.RPN_PRE_NMS_TOP_N_TRAIN}")
    print(f"  RPN post-NMS top N (train): {Config.RPN_POST_NMS_TOP_N_TRAIN}")

    # Configure ROI head
    model.roi_heads.box_roi_pool.output_size = (Config.ROI_OUTPUT_SIZE, Config.ROI_OUTPUT_SIZE)
    model.roi_heads.nms_thresh = Config.ROI_NMS_THRESH
    model.roi_heads.score_thresh = Config.ROI_SCORE_THRESH
    model.roi_heads.batch_size_per_image = Config.ROI_BATCH_SIZE_PER_IMAGE
    model.roi_heads.positive_fraction = Config.ROI_POSITIVE_FRACTION
    print(f"  ROI output size: {Config.ROI_OUTPUT_SIZE}x{Config.ROI_OUTPUT_SIZE}")
    print(f"  ROI NMS threshold: {Config.ROI_NMS_THRESH}")
    print(f"  ROI score threshold: {Config.ROI_SCORE_THRESH}")

    # Update classifier for custom number of classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    print(f"  Updated classifier: {in_features} -> {num_classes}")

    print("\nModel created successfully!")
    return model


def print_model_summary(model):
    """
    Prints detailed summary of the model.

    Parameters:
    - model: PyTorch model to summarize
    """
    print("\n" + "="*80)
    print("MODEL SUMMARY")
    print("="*80)

    try:
        # Create dummy input
        dummy_input = torch.randn(1, 3, Config.IMG_SIZE, Config.IMG_SIZE)

        # Get model summary using torchinfo
        summary(
            model,
            input_data=[dummy_input],
            col_names=["input_size", "output_size", "num_params", "trainable"],
            col_width=20,
            row_settings=["var_names"],
            depth=3  # Show only top-level modules
        )

        # Print additional statistics
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        backbone_params = sum(p.numel() for p in model.backbone.parameters())
        rpn_params = sum(p.numel() for p in model.rpn.parameters())
        roi_params = sum(p.numel() for p in model.roi_heads.parameters())

        print("\n" + "-"*80)
        print("Parameter Breakdown:")
        print(f"  Total Parameters: {total_params:,}")
        print(f"  Trainable Parameters: {trainable_params:,}")
        print(f"  Non-trainable Parameters: {total_params - trainable_params:,}")
        print(f"  Backbone Parameters: {backbone_params:,}")
        print(f"  RPN Parameters: {rpn_params:,}")
        print(f"  ROI Head Parameters: {roi_params:,}")
        print("="*80)

    except Exception as e:
        print(f"Error generating summary: {e}")
        print("\nFallback to basic parameter count:")
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total Parameters: {total_params:,}")


# Create model
num_classes = len(train_dataset.classes)
model = create_model(num_classes)

# Move model to device
model = model.to(Config.DEVICE)

# Print model summary
print_model_summary(model)

# Print model architecture details
print("\nModel Architecture Details:")
print("="*80)
print(f"  Backbone: {Config.BACKBONE}")
print(f"  Anchor Sizes: {Config.ANCHOR_SIZES}")
print(f"  Aspect Ratios: {Config.ASPECT_RATIOS}")
print(f"  ROI Output Size: {Config.ROI_OUTPUT_SIZE}")
print(f"  Device: {Config.DEVICE}")
print("="*80)

Creating Faster R-CNN model...
  Number of classes: 8
  Using ResNet50 backbone (balanced)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 159MB/s]


  Custom anchors: ((16, 32, 64, 128, 256),)
  RPN NMS threshold: 0.7
  RPN pre-NMS top N (train): 2000
  RPN post-NMS top N (train): 1000
  ROI output size: 14x14
  ROI NMS threshold: 0.5
  ROI score threshold: 0.05
  Updated classifier: 1024 -> 8

Model created successfully!

MODEL SUMMARY
Error generating summary: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [GeneralizedRCNNTransform: 1]

Fallback to basic parameter count:
Total Parameters: 41,329,911

Model Architecture Details:
  Backbone: resnet50
  Anchor Sizes: ((16, 32, 64, 128, 256),)
  Aspect Ratios: ((0.5, 1.0, 2.0),)
  ROI Output Size: 14
  Device: cuda


In [10]:
# ============================================
# BLOCK 5: Optimizer, Scheduler, and Training Setup
# ============================================

def setup_optimizer(model):
    """
    Sets up optimizer with proper parameter groups.

    Parameters:
    - model: PyTorch model

    Returns:
    - Configured optimizer
    """
    # Separate backbone parameters (lower learning rate) from others
    backbone_params = []
    other_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'backbone' in name:
            backbone_params.append(param)
        else:
            other_params.append(param)

    print(f"Optimizer setup:")
    print(f"  Backbone parameters: {len(backbone_params)} groups")
    print(f"  Other parameters: {len(other_params)} groups")

    # Create parameter groups with different learning rates
    params = [
        {'params': backbone_params, 'lr': Config.LEARNING_RATE * 0.1},  # Lower LR for backbone
        {'params': other_params, 'lr': Config.LEARNING_RATE}
    ]

    # SGD optimizer with momentum
    optimizer = optim.SGD(
        params,
        lr=Config.LEARNING_RATE,
        momentum=Config.MOMENTUM,
        weight_decay=Config.WEIGHT_DECAY,
        nesterov=True  # Use Nesterov momentum for better convergence
    )

    print(f"  Optimizer: SGD with Nesterov momentum")
    print(f"  Base learning rate: {Config.LEARNING_RATE}")
    print(f"  Backbone learning rate: {Config.LEARNING_RATE * 0.1}")
    print(f"  Momentum: {Config.MOMENTUM}")
    print(f"  Weight decay: {Config.WEIGHT_DECAY}")

    return optimizer


def setup_scheduler(optimizer):
    """
    Sets up learning rate scheduler based on configuration.

    Parameters:
    - optimizer: PyTorch optimizer

    Returns:
    - Configured scheduler
    """
    print(f"\nScheduler setup:")
    print(f"  Type: {Config.LR_SCHEDULER}")

    if Config.LR_SCHEDULER == 'step':
        scheduler = optim.lr_scheduler.StepLR(
            optimizer,
            step_size=Config.STEP_SIZE,
            gamma=Config.GAMMA
        )
        print(f"  Step size: {Config.STEP_SIZE}")
        print(f"  Gamma: {Config.GAMMA}")

    elif Config.LR_SCHEDULER == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=Config.EPOCHS,
            eta_min=Config.LEARNING_RATE * 0.01
        )
        print(f"  T_max: {Config.EPOCHS}")
        print(f"  Min LR: {Config.LEARNING_RATE * 0.01}")

    elif Config.LR_SCHEDULER == 'reduce_on_plateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=3,
            verbose=True
        )
        print(f"  Factor: 0.5")
        print(f"  Patience: 3")

    else:
        raise ValueError(f"Unknown scheduler: {Config.LR_SCHEDULER}")

    return scheduler


def setup_training():
    """
    Sets up all training components: optimizer, scheduler, scaler, and writer.

    Returns:
    - optimizer: Configured optimizer
    - scheduler: Configured scheduler
    - scaler: GradScaler for mixed precision
    - writer: TensorBoard writer
    """
    print("="*80)
    print("TRAINING SETUP")
    print("="*80)

    # Setup optimizer
    optimizer = setup_optimizer(model)

    # Setup scheduler
    scheduler = setup_scheduler(optimizer)

    # Setup GradScaler for mixed precision training
    scaler = GradScaler(enabled=Config.USE_AMP)
    print(f"\nMixed Precision Training:")
    print(f"  Enabled: {Config.USE_AMP}")

    # Setup TensorBoard writer
    writer = SummaryWriter('runs/aquarium_detection')
    print(f"\nTensorBoard:")
    print(f"  Log directory: runs/aquarium_detection")
    print(f"  To view: tensorboard --logdir=runs")

    print("\n" + "="*80)
    print("Training setup complete!")
    print("="*80)

    return optimizer, scheduler, scaler, writer


# Setup all training components
optimizer, scheduler, scaler, writer = setup_training()

# Print a sample of parameter groups
print("\nParameter groups:")
for i, group in enumerate(optimizer.param_groups):
    print(f"  Group {i}: LR={group['lr']:.6f}, Size={len(group['params'])}")

TRAINING SETUP
Optimizer setup:
  Backbone parameters: 58 groups
  Other parameters: 14 groups
  Optimizer: SGD with Nesterov momentum
  Base learning rate: 0.005
  Backbone learning rate: 0.0005
  Momentum: 0.9
  Weight decay: 0.0005

Scheduler setup:
  Type: cosine
  T_max: 15
  Min LR: 5e-05

Mixed Precision Training:
  Enabled: True

TensorBoard:
  Log directory: runs/aquarium_detection
  To view: tensorboard --logdir=runs

Training setup complete!

Parameter groups:
  Group 0: LR=0.000500, Size=58
  Group 1: LR=0.005000, Size=14


In [11]:
# ============================================
# BLOCK 6: Training Function with All Improvements
# ============================================

def train_one_epoch(model, optimizer, scaler, loader, device, epoch, writer=None):
    """
    Trains the model for one epoch with mixed precision, gradient clipping,
    and comprehensive loss tracking.

    Parameters:
    - model: neural network model
    - optimizer: optimizer for backpropagation
    - scaler: GradScaler for mixed precision
    - loader: dataloader for training data
    - device: device to run computations (CPU/GPU)
    - epoch: current epoch number
    - writer: TensorBoard writer (optional)

    Returns:
    - Dictionary containing average losses for the epoch
    """
    model.train()

    # Initialize loss tracking
    total_loss = 0.0
    loss_dict_accumulated = {
        'loss_classifier': 0.0,
        'loss_box_reg': 0.0,
        'loss_objectness': 0.0,
        'loss_rpn_box_reg': 0.0
    }

    num_batches = len(loader)

    # Progress bar
    progress_bar = tqdm(loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS}')

    for batch_idx, (images, targets) in enumerate(progress_bar):
        # Move data to device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass with mixed precision
        if Config.USE_AMP:
            with autocast():
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())

            # Backward pass with scaler
            scaler.scale(losses).backward()

            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)

            # Optimizer step with scaler
            scaler.step(optimizer)
            scaler.update()
        else:
            # Without mixed precision
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            # Backward pass
            losses.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)

            # Optimizer step
            optimizer.step()

        # Accumulate losses
        total_loss += losses.item()
        for key, loss_val in loss_dict.items():
            if key in loss_dict_accumulated:
                loss_dict_accumulated[key] += loss_val.item()

        # Update progress bar
        if batch_idx % 10 == 0:
            progress_bar.set_postfix({
                'loss': f'{losses.item():.4f}',
                'cls': f'{loss_dict.get("loss_classifier", 0):.3f}',
                'box': f'{loss_dict.get("loss_box_reg", 0):.3f}'
            })

        # Log to TensorBoard (every 10 batches)
        if writer and batch_idx % 10 == 0:
            global_step = epoch * num_batches + batch_idx
            writer.add_scalar('Train/Loss_batch', losses.item(), global_step)
            for key, val in loss_dict.items():
                if key in loss_dict_accumulated:
                    writer.add_scalar(f'Train/{key}', val.item(), global_step)

    # Calculate average losses
    avg_total_loss = total_loss / num_batches
    avg_loss_dict = {key: val / num_batches for key, val in loss_dict_accumulated.items()}

    # Print epoch summary
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Total Loss: {avg_total_loss:.4f}")
    for loss_name, avg_loss in avg_loss_dict.items():
        print(f"  {loss_name}: {avg_loss:.4f}")

    # Log to TensorBoard
    if writer:
        writer.add_scalar('Train/Loss_epoch', avg_total_loss, epoch)
        for key, val in avg_loss_dict.items():
            writer.add_scalar(f'Train/{key}_epoch', val, epoch)
        writer.add_scalar('Train/Learning_Rate', optimizer.param_groups[0]['lr'], epoch)

    return avg_loss_dict


def validate_one_epoch(model, loader, device, epoch, writer=None):
    """
    Validates the model on validation set.

    Parameters:
    - model: neural network model
    - loader: dataloader for validation data
    - device: device to run computations
    - epoch: current epoch number
    - writer: TensorBoard writer (optional)

    Returns:
    - Dictionary containing average validation losses
    """
    model.eval()

    # Initialize loss tracking
    total_loss = 0.0
    loss_dict_accumulated = {
        'loss_classifier': 0.0,
        'loss_box_reg': 0.0,
        'loss_objectness': 0.0,
        'loss_rpn_box_reg': 0.0
    }

    num_batches = len(loader)

    with torch.no_grad():
        progress_bar = tqdm(loader, desc=f'Validation Epoch {epoch+1}')

        for images, targets in progress_bar:
            # Move data to device
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Forward pass
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            # Accumulate losses
            total_loss += losses.item()
            for key, loss_val in loss_dict.items():
                if key in loss_dict_accumulated:
                    loss_dict_accumulated[key] += loss_val.item()

            # Update progress bar
            progress_bar.set_postfix({'val_loss': f'{losses.item():.4f}'})

    # Calculate average losses
    avg_total_loss = total_loss / num_batches
    avg_loss_dict = {key: val / num_batches for key, val in loss_dict_accumulated.items()}

    # Print validation summary
    print(f"\nValidation Epoch {epoch+1} Summary:")
    print(f"  Total Loss: {avg_total_loss:.4f}")
    for loss_name, avg_loss in avg_loss_dict.items():
        print(f"  {loss_name}: {avg_loss:.4f}")

    # Log to TensorBoard
    if writer:
        writer.add_scalar('Validation/Loss_epoch', avg_total_loss, epoch)
        for key, val in avg_loss_dict.items():
            writer.add_scalar(f'Validation/{key}_epoch', val, epoch)

    return avg_loss_dict


def save_checkpoint(model, optimizer, scheduler, epoch, loss, best_loss, path='checkpoint.pth'):
    """
    Saves model checkpoint.

    Parameters:
    - model: model to save
    - optimizer: optimizer state
    - scheduler: scheduler state
    - epoch: current epoch
    - loss: current loss
    - best_loss: best loss so far
    - path: save path
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'loss': loss,
        'best_loss': best_loss,
        'config': {
            'backbone': Config.BACKBONE,
            'img_size': Config.IMG_SIZE,
            'num_classes': len(train_dataset.classes)
        }
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path}")


def load_checkpoint(model, optimizer, scheduler, path='checkpoint.pth'):
    """
    Loads model checkpoint.

    Parameters:
    - model: model to load weights into
    - optimizer: optimizer to load state
    - scheduler: scheduler to load state
    - path: checkpoint path

    Returns:
    - epoch, loss, best_loss
    """
    checkpoint = torch.load(path, map_location=Config.DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    best_loss = checkpoint['best_loss']
    print(f"Checkpoint loaded from {path}")
    print(f"  Epoch: {epoch}, Loss: {loss:.4f}, Best Loss: {best_loss:.4f}")
    return epoch, loss, best_loss

In [15]:
print(f"Train batches: {len(train_loader)}")   # 112
print(f"Valid batches: {len(valid_loader)}")   # 32
print(f"Test batches: {len(test_loader)}")     # 16

Train batches: 112
Valid batches: 32
Test batches: 16


In [16]:
# Check validation dataset
print(f"Validation dataset size: {len(valid_dataset)}")

# Check first few samples
for i in range(min(3, len(valid_dataset))):
    try:
        img, target = valid_dataset[i]
        print(f"\nSample {i}:")
        print(f"  Image shape: {img.shape}")
        print(f"  Number of boxes: {len(target['boxes'])}")
        if len(target['boxes']) > 0:
            print(f"  Boxes: {target['boxes'][:2]}")
        print(f"  Labels: {target['labels']}")
    except Exception as e:
        print(f"  Error loading sample {i}: {e}")

Validation dataset size: 127

Sample 0:
  Image shape: torch.Size([3, 640, 640])
  Number of boxes: 1
  Boxes: tensor([[115.0000, 197.5000, 440.8333, 425.0000]])
  Labels: tensor([6])

Sample 1:
  Image shape: torch.Size([3, 640, 640])
  Number of boxes: 4
  Boxes: tensor([[105.0000, 246.8750, 183.3333, 360.0000],
        [275.8333, 475.6250, 639.1667, 577.5000]])
  Labels: tensor([1, 5, 6, 6])

Sample 2:
  Image shape: torch.Size([3, 640, 640])
  Number of boxes: 1
  Boxes: tensor([[160.0000, 238.3333, 225.0000, 383.3333]])
  Labels: tensor([1])


In [17]:
# Check if validation loader has data
try:
    val_images, val_targets = next(iter(valid_loader))
    print(f"Validation batch loaded successfully!")
    print(f"  Number of images: {len(val_images)}")
    print(f"  Number of targets: {len(val_targets)}")
    print(f"  First image shape: {val_images[0].shape}")
    print(f"  First target boxes: {val_targets[0]['boxes'].shape}")
except StopIteration:
    print("Validation loader is empty!")
except Exception as e:
    print(f"Error loading validation batch: {e}")

Validation batch loaded successfully!
  Number of images: 4
  Number of targets: 4
  First image shape: torch.Size([3, 640, 640])
  First target boxes: torch.Size([1, 4])


In [21]:
# ============================================
# DEBUG: Check What Model Returns
# ============================================

print("="*80)
print("DEBUG: CHECK MODEL OUTPUT")
print("="*80)

# Get a batch
val_batch = next(iter(valid_loader))
val_images, val_targets = val_batch

# Move to device
images = [img.to(Config.DEVICE) for img in val_images]

# Create targets
device_targets = []
for t in val_targets:
    device_target = {
        'boxes': t['boxes'].to(Config.DEVICE),
        'labels': t['labels'].to(Config.DEVICE),
        'image_id': t['image_id'].to(Config.DEVICE),
        'area': t['area'].to(Config.DEVICE) if len(t['area']) > 0 else t['area'],
        'iscrowd': t['iscrowd'].to(Config.DEVICE) if len(t['iscrowd']) > 0 else t['iscrowd']
    }
    device_targets.append(device_target)

print(f"Images: {len(images)}")
print(f"Targets: {len(device_targets)}")
print(f"Target 0 keys: {device_targets[0].keys()}")

# Check model forward pass
model.eval()
with torch.no_grad():
    print("\nCalling model(images, targets)...")
    result = model(images, device_targets)
    print(f"Result type: {type(result)}")
    print(f"Result: {result}")

    if isinstance(result, dict):
        print(f"Result keys: {result.keys()}")
        loss = sum(result.values())
        print(f"Loss: {loss.item():.4f}")
    elif isinstance(result, list):
        print(f"Result is a list of length: {len(result)}")
        for i, item in enumerate(result):
            print(f"  Item {i}: {type(item)}")
            if isinstance(item, dict):
                print(f"    Keys: {item.keys()}")
    else:
        print(f"Unexpected result type: {type(result)}")

DEBUG: CHECK MODEL OUTPUT
Images: 4
Targets: 4
Target 0 keys: dict_keys(['boxes', 'labels', 'image_id', 'area', 'iscrowd'])

Calling model(images, targets)...
Result type: <class 'list'>
Result: [{'boxes': tensor([[1.0490e+02, 2.1073e+02, 4.4504e+02, 4.0195e+02],
        [1.6327e+02, 5.9140e+01, 3.6828e+02, 1.4165e+02],
        [1.1161e+02, 2.1359e+02, 4.7636e+02, 4.1633e+02],
        [4.3393e+02, 2.5396e-01, 6.1542e+02, 1.6951e+02],
        [8.9922e+01, 1.7687e+02, 2.6591e+02, 2.4289e+02],
        [4.3884e+02, 0.0000e+00, 6.1585e+02, 1.5551e+02],
        [1.2332e+01, 4.2544e+02, 1.3444e+02, 4.5468e+02],
        [9.8471e+01, 1.9844e+02, 2.4515e+02, 2.3707e+02],
        [4.2062e+02, 5.1486e+00, 6.1692e+02, 1.6345e+02]], device='cuda:0'), 'labels': tensor([6, 1, 7, 1, 1, 5, 1, 1, 7], device='cuda:0'), 'scores': tensor([0.8265, 0.2654, 0.1682, 0.1242, 0.1008, 0.0831, 0.0573, 0.0535, 0.0503],
       device='cuda:0')}, {'boxes': tensor([[358.0892, 414.2075, 492.6522, 487.4813],
        [101

In [23]:
# ============================================
# FIX: Correct Validation with Train Mode
# ============================================

print("="*80)
print("FIX: CORRECT VALIDATION WITH TRAIN MODE")
print("="*80)

# ============================================
# CORRECT VALIDATION FUNCTION
# ============================================

def validate_correct_final(model, loader, device):
    """
    Correct validation function.
    IMPORTANT: Model must be in train() mode to compute loss,
    but we use torch.no_grad() to prevent gradient computation.
    """
    # Set model to train mode (to compute loss)
    model.train()
    total_loss = 0.0
    successful_batches = 0
    total_batches = len(loader)

    print(f"\nValidating on {total_batches} batches...")

    with torch.no_grad():  # No gradient computation
        for batch_idx, (images, targets) in enumerate(loader):
            try:
                # Skip empty batch
                if len(images) == 0:
                    continue

                # Move images to device
                images = [img.to(device) for img in images]

                # Move targets to device
                device_targets = []
                for t in targets:
                    device_target = {
                        'boxes': t['boxes'].to(device),
                        'labels': t['labels'].to(device),
                        'image_id': t['image_id'].to(device),
                        'area': t['area'].to(device) if len(t['area']) > 0 else t['area'],
                        'iscrowd': t['iscrowd'].to(device) if len(t['iscrowd']) > 0 else t['iscrowd']
                    }
                    device_targets.append(device_target)

                # Check if any target has boxes
                has_boxes = any(len(t['boxes']) > 0 for t in device_targets)
                if not has_boxes:
                    if batch_idx < 3:
                        print(f"  Batch {batch_idx}: No boxes, skipping")
                    continue

                # Forward pass - model in train mode returns loss dict
                loss_dict = model(images, device_targets)

                # Check if we got a dict (loss) or list (predictions)
                if isinstance(loss_dict, dict):
                    loss = sum(loss_dict.values())
                    total_loss += loss.item()
                    successful_batches += 1

                    if batch_idx % 5 == 0:
                        print(f"  Batch {batch_idx}: Loss = {loss.item():.4f}")
                        print(f"    Loss components: { {k: v.item() for k, v in loss_dict.items()} }")
                else:
                    print(f"  Batch {batch_idx}: Got predictions instead of loss! Model may not be in train mode.")
                    continue

            except Exception as e:
                if batch_idx < 3:
                    print(f"  Batch {batch_idx}: Error - {str(e)[:100]}")
                continue

    if successful_batches > 0:
        avg_loss = total_loss / successful_batches
        print(f"\n✅ Validation Complete!")
        print(f"   Successful batches: {successful_batches}/{total_batches}")
        print(f"   Average Loss: {avg_loss:.4f}")
        return avg_loss
    else:
        print(f"\n❌ Validation Failed!")
        print(f"   No successful batches out of {total_batches}")
        return float('inf')


# ============================================
# CORRECT TRAINING FUNCTION
# ============================================

def train_correct_final(model, optimizer, scaler, loader, device, epoch):
    """
    Correct training function.
    """
    # Set model to train mode
    model.train()
    total_loss = 0.0
    successful_batches = 0

    progress_bar = tqdm(loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS}')

    for images, targets in progress_bar:
        try:
            # Move images to device
            images = [img.to(device) for img in images]

            # Move targets to device
            device_targets = []
            for t in targets:
                device_target = {
                    'boxes': t['boxes'].to(device),
                    'labels': t['labels'].to(device),
                    'image_id': t['image_id'].to(device),
                    'area': t['area'].to(device) if len(t['area']) > 0 else t['area'],
                    'iscrowd': t['iscrowd'].to(device) if len(t['iscrowd']) > 0 else t['iscrowd']
                }
                device_targets.append(device_target)

            # Skip if no boxes
            if all(len(t['boxes']) == 0 for t in device_targets):
                continue

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass with AMP
            with torch.cuda.amp.autocast():
                loss_dict = model(images, device_targets)
                # In train mode, loss_dict should be a dict
                if isinstance(loss_dict, dict):
                    loss = sum(loss_dict.values())
                else:
                    print(f"Warning: Got predictions in train mode!")
                    continue

            # Backward pass
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            successful_batches += 1

            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg': f'{total_loss/successful_batches:.4f}'
            })

        except Exception as e:
            if successful_batches % 20 == 0:
                print(f"Warning: Batch error - {str(e)[:50]}")
            continue

    avg_loss = total_loss / successful_batches if successful_batches > 0 else 0
    return avg_loss, successful_batches


# ============================================
# TEST VALIDATION
# ============================================

print("\nTesting validation function...")

# Make sure model is on correct device
model = model.to(Config.DEVICE)

# Test validation
val_loss = validate_correct_final(model, valid_loader, Config.DEVICE)

# Print result safely
if val_loss != float('inf'):
    print(f"\n✅ Validation Loss: {val_loss:.4f}")
else:
    print(f"\n❌ Validation Loss: N/A")

# ============================================
# RUN FULL TRAINING
# ============================================

print("\n" + "="*80)
print("RUNNING FULL TRAINING WITH CORRECT CODE")
print("="*80)

# Recreate model if needed
num_classes = len(train_dataset.classes)
print(f"Number of classes: {num_classes}")

# Create fresh model
model = models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model = model.to(Config.DEVICE)

# Optimizer and scheduler
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=Config.LEARNING_RATE,
    momentum=Config.MOMENTUM,
    weight_decay=Config.WEIGHT_DECAY,
    nesterov=True
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=Config.EPOCHS,
    eta_min=Config.LEARNING_RATE * 0.01
)

scaler = torch.cuda.amp.GradScaler()

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rates': []
}

best_val_loss = float('inf')
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Run training
for epoch in range(Config.EPOCHS):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch+1}/{Config.EPOCHS}")
    print(f"{'='*80}")

    # Training
    train_loss, train_count = train_correct_final(
        model, optimizer, scaler, train_loader, Config.DEVICE, epoch
    )

    # Validation - model stays in train mode but with no_grad
    val_loss = validate_correct_final(model, valid_loader, Config.DEVICE)

    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rates'].append(current_lr)

    # Print results
    print(f"\nEpoch {epoch+1} Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Train Batches: {train_count}/{len(train_loader)}")
    if val_loss != float('inf'):
        print(f"  ✅ Validation Loss: {val_loss:.4f}")
    else:
        print(f"  ❌ Validation Loss: N/A")
    print(f"  Learning Rate: {current_lr:.6f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  ✅ New best model saved! Val Loss: {best_val_loss:.4f}")

    # Save checkpoint every 3 epochs
    if (epoch + 1) % 3 == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'history': history
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth'))
        print(f"  Checkpoint saved to checkpoint_epoch_{epoch+1}.pth")

# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)

valid_val_losses = [v for v in history['val_loss'] if v != float('inf')]

print(f"\nTraining Results:")
print(f"  Best Train Loss: {min(history['train_loss']):.4f}")
print(f"  Final Train Loss: {history['train_loss'][-1]:.4f}")
if valid_val_losses:
    print(f"  ✅ Best Validation Loss: {min(valid_val_losses):.4f}")
    print(f"  ✅ Final Validation Loss: {history['val_loss'][-1]:.4f}")
else:
    print(f"  ❌ Validation Loss: N/A")

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)

FIX: CORRECT VALIDATION WITH TRAIN MODE

Testing validation function...

Validating on 32 batches...
  Batch 0: Loss = 0.2102
    Loss components: {'loss_classifier': 0.087810218334198, 'loss_box_reg': 0.07402979582548141, 'loss_objectness': 0.042156293988227844, 'loss_rpn_box_reg': 0.006187424995005131}
  Batch 5: Loss = 0.6460
    Loss components: {'loss_classifier': 0.27117300033569336, 'loss_box_reg': 0.2969333827495575, 'loss_objectness': 0.06269654631614685, 'loss_rpn_box_reg': 0.015230846591293812}
  Batch 10: Loss = 0.5205
    Loss components: {'loss_classifier': 0.21318885684013367, 'loss_box_reg': 0.2737348675727844, 'loss_objectness': 0.01635684445500374, 'loss_rpn_box_reg': 0.01723451539874077}
  Batch 15: Loss = 0.6105
    Loss components: {'loss_classifier': 0.21880030632019043, 'loss_box_reg': 0.33620208501815796, 'loss_objectness': 0.031552691012620926, 'loss_rpn_box_reg': 0.02392391860485077}
  Batch 20: Loss = 0.5558
    Loss components: {'loss_classifier': 0.28949838

Epoch 1/15: 100%|██████████| 112/112 [00:53<00:00,  2.11it/s, loss=0.8105, avg=1.1639]



Validating on 32 batches...
  Batch 0: Loss = 0.3585
    Loss components: {'loss_classifier': 0.18349754810333252, 'loss_box_reg': 0.1450776755809784, 'loss_objectness': 0.023543059825897217, 'loss_rpn_box_reg': 0.006410886533558369}
  Batch 5: Loss = 0.7646
    Loss components: {'loss_classifier': 0.35195356607437134, 'loss_box_reg': 0.3565281331539154, 'loss_objectness': 0.04041224718093872, 'loss_rpn_box_reg': 0.015659017488360405}
  Batch 10: Loss = 0.7099
    Loss components: {'loss_classifier': 0.3315962553024292, 'loss_box_reg': 0.3307103216648102, 'loss_objectness': 0.029096025973558426, 'loss_rpn_box_reg': 0.018451698124408722}
  Batch 15: Loss = 0.8196
    Loss components: {'loss_classifier': 0.33430176973342896, 'loss_box_reg': 0.4198272228240967, 'loss_objectness': 0.04279681295156479, 'loss_rpn_box_reg': 0.022635875269770622}
  Batch 20: Loss = 0.7882
    Loss components: {'loss_classifier': 0.3706508278846741, 'loss_box_reg': 0.3536292612552643, 'loss_objectness': 0.0401

Epoch 2/15: 100%|██████████| 112/112 [00:52<00:00,  2.12it/s, loss=0.5621, avg=0.7358]



Validating on 32 batches...
  Batch 0: Loss = 0.2943
    Loss components: {'loss_classifier': 0.14428597688674927, 'loss_box_reg': 0.11669029295444489, 'loss_objectness': 0.026729676872491837, 'loss_rpn_box_reg': 0.006584964692592621}
  Batch 5: Loss = 0.6306
    Loss components: {'loss_classifier': 0.2855634391307831, 'loss_box_reg': 0.2918168604373932, 'loss_objectness': 0.03468114137649536, 'loss_rpn_box_reg': 0.018502214923501015}
  Batch 10: Loss = 0.5941
    Loss components: {'loss_classifier': 0.26670563220977783, 'loss_box_reg': 0.294440895318985, 'loss_objectness': 0.014496469870209694, 'loss_rpn_box_reg': 0.018455082550644875}
  Batch 15: Loss = 0.6618
    Loss components: {'loss_classifier': 0.24893546104431152, 'loss_box_reg': 0.3554806709289551, 'loss_objectness': 0.034583598375320435, 'loss_rpn_box_reg': 0.02283737249672413}
  Batch 20: Loss = 0.6189
    Loss components: {'loss_classifier': 0.3020820617675781, 'loss_box_reg': 0.266013503074646, 'loss_objectness': 0.02895

...


Validating on 32 batches...
  Batch 0: Loss = 0.2192
    Loss components: {'loss_classifier': 0.09189189225435257, 'loss_box_reg': 0.07315978407859802, 'loss_objectness': 0.049042023718357086, 'loss_rpn_box_reg': 0.005129213444888592}
  Batch 5: Loss = 0.6455
    Loss components: {'loss_classifier': 0.24802479147911072, 'loss_box_reg': 0.2759329080581665, 'loss_objectness': 0.10478048771619797, 'loss_rpn_box_reg': 0.016781993210315704}
  Batch 10: Loss = 0.5381
    Loss components: {'loss_classifier': 0.2305505871772766, 'loss_box_reg': 0.27066463232040405, 'loss_objectness': 0.020848771557211876, 'loss_rpn_box_reg': 0.016000375151634216}
  Batch 15: Loss = 0.6665
    Loss components: {'loss_classifier': 0.27775096893310547, 'loss_box_reg': 0.32743173837661743, 'loss_objectness': 0.03717542439699173, 'loss_rpn_box_reg': 0.024106435477733612}
  Batch 20: Loss = 0.6240
    Loss components: {'loss_classifier': 0.3214440941810608, 'loss_box_reg': 0.24815377593040466, 'loss_objectness': 0.

Epoch 14/15: 100%|██████████| 112/112 [00:52<00:00,  2.13it/s, loss=0.0502, avg=0.2579]



Validating on 32 batches...
  Batch 0: Loss = 0.2210
    Loss components: {'loss_classifier': 0.08751969039440155, 'loss_box_reg': 0.07755877077579498, 'loss_objectness': 0.050671495497226715, 'loss_rpn_box_reg': 0.005244780797511339}
  Batch 5: Loss = 0.6257
    Loss components: {'loss_classifier': 0.2405150830745697, 'loss_box_reg': 0.26002609729766846, 'loss_objectness': 0.10839998722076416, 'loss_rpn_box_reg': 0.01674039661884308}
  Batch 10: Loss = 0.5180
    Loss components: {'loss_classifier': 0.22301025688648224, 'loss_box_reg': 0.2609511911869049, 'loss_objectness': 0.01806565560400486, 'loss_rpn_box_reg': 0.01602279767394066}
  Batch 15: Loss = 0.6424
    Loss components: {'loss_classifier': 0.2675403952598572, 'loss_box_reg': 0.3211033344268799, 'loss_objectness': 0.029486577957868576, 'loss_rpn_box_reg': 0.02429851144552231}
  Batch 20: Loss = 0.6053
    Loss components: {'loss_classifier': 0.30733293294906616, 'loss_box_reg': 0.24519622325897217, 'loss_objectness': 0.0333

Epoch 15/15: 100%|██████████| 112/112 [00:53<00:00,  2.11it/s, loss=0.2265, avg=0.2549]



Validating on 32 batches...
  Batch 0: Loss = 0.2178
    Loss components: {'loss_classifier': 0.08886246383190155, 'loss_box_reg': 0.07461840659379959, 'loss_objectness': 0.04912913590669632, 'loss_rpn_box_reg': 0.005169184412807226}
  Batch 5: Loss = 0.6427
    Loss components: {'loss_classifier': 0.23973529040813446, 'loss_box_reg': 0.27827727794647217, 'loss_objectness': 0.10803051292896271, 'loss_rpn_box_reg': 0.016691621392965317}
  Batch 10: Loss = 0.5343
    Loss components: {'loss_classifier': 0.2265835851430893, 'loss_box_reg': 0.2734498977661133, 'loss_objectness': 0.018206246197223663, 'loss_rpn_box_reg': 0.01601129025220871}
  Batch 15: Loss = 0.6287
    Loss components: {'loss_classifier': 0.2534983456134796, 'loss_box_reg': 0.3226351737976074, 'loss_objectness': 0.028244860470294952, 'loss_rpn_box_reg': 0.024304402992129326}
  Batch 20: Loss = 0.5964
    Loss components: {'loss_classifier': 0.299480140209198, 'loss_box_reg': 0.24282358586788177, 'loss_objectness': 0.0347